# Notebook n°1 : Qualité des données

### Problématique

> **Dans quelle mesure la structure du réseau TGV est-elle associée à la régularité des liaisons, et peut-on exploiter cette structure pour améliorer la prédiction des retards ?**

## 1. Objectifs de ce notebook

Ce premier notebook est consacré à l'évaluation de la **qualité des données SNCF** avant leur exploitation dans les analyses statistiques et les modèles prédictifs.

Les objectifs sont les suivants :

- Examiner la structure et les dimensions du jeu de données.
- Identifier les types des différentes variables.
- Quantifier les valeurs manquantes.
- Rechercher les doublons et vérifier l'unicité des observations.
- Contrôler la cohérence des circulations, des annulations et des retards.
- Vérifier les durées moyennes et les pourcentages des causes de retard.
- Identifier et étudier les observations atypiques ou incohérentes.
- Formuler des recommandations pour la préparation des données.

Cette étape permet de limiter les risques de biais et d'erreurs lors des analyses et de la modélisation.

Les fonctions sur le contrôle de la qualité des données sont répertoriés dans le fichier quality.py. Les résultats sont regroupés dans un dictionnaire via report.py. Toutes les fonctions possèdents des tests unitaires.

# I. Chargement des données depuis l'API

In [53]:
"""Chargement des données SNCF depuis l'API."""

from io import BytesIO
from pathlib import Path
import sys

import pandas as pd
import requests

# Définir la racine du projet.
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Permettre l'import des fonctions de notre projet.
sys.path.insert(0, str(project_root / "src"))

from sncf_tgv.eda.report import generate_quality_report

# URL de l'API SNCF.
API_URL = (
    "https://ressources.data.sncf.com/api/explore/v2.1/"
    "catalog/datasets/regularite-mensuelle-tgv-aqst/exports/csv"
)

# Récupérer les données.
response = requests.get(API_URL, timeout=60)
response.raise_for_status()

# Charger les données dans un DataFrame.
data = pd.read_csv(BytesIO(response.content), sep=";")

# Vérifier le chargement.
print("Dimensions :", data.shape)
display(data.head())


Dimensions : (12907, 26)


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation,commentaire_annulation,nb_train_depart_retard,retard_moyen_depart,...,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15,nb_train_retard_sup_30,nb_train_retard_sup_60,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


# II. Analyse de la qualité du jeu de données

## 1. Structure et dimensions du jeu de données

In [54]:
print("Dimensions :", data.shape)
print("Colonnes :", data.columns.tolist())

Dimensions : (12907, 26)
Colonnes : ['date', 'service', 'gare_depart', 'gare_arrivee', 'duree_moyenne', 'nb_train_prevu', 'nb_annulation', 'commentaire_annulation', 'nb_train_depart_retard', 'retard_moyen_depart', 'retard_moyen_tous_trains_depart', 'commentaire_retards_depart', 'nb_train_retard_arrivee', 'retard_moyen_arrivee', 'retard_moyen_tous_trains_arrivee', 'commentaires_retard_arrivee', 'nb_train_retard_sup_15', 'retard_moyen_trains_retard_sup15', 'nb_train_retard_sup_30', 'nb_train_retard_sup_60', 'prct_cause_externe', 'prct_cause_infra', 'prct_cause_gestion_trafic', 'prct_cause_materiel_roulant', 'prct_cause_gestion_gare', 'prct_cause_prise_en_charge_voyageurs']


## 2. Types des différentes variables

In [55]:
print("Types :", data.dtypes)

Types : date                                        str
service                                     str
gare_depart                                 str
gare_arrivee                                str
duree_moyenne                             int64
nb_train_prevu                            int64
nb_annulation                             int64
commentaire_annulation                  float64
nb_train_depart_retard                    int64
retard_moyen_depart                     float64
retard_moyen_tous_trains_depart         float64
commentaire_retards_depart              float64
nb_train_retard_arrivee                   int64
retard_moyen_arrivee                    float64
retard_moyen_tous_trains_arrivee        float64
commentaires_retard_arrivee                 str
nb_train_retard_sup_15                    int64
retard_moyen_trains_retard_sup15        float64
nb_train_retard_sup_30                    int64
nb_train_retard_sup_60                    int64
prct_cause_externe              

## 3.  Quantification des valeurs manquantes et des doublons

In [56]:

report = generate_quality_report(data)
print(f"""
Rapport sur la qualité des données:
Nombre de lignes: {report["overview"]["n_rows"]}
Nombre de colonnes: {report["overview"]["n_columns"]}
Doublons : {report["duplicates"]["complete_rows"]}
""")





Rapport sur la qualité des données:
Nombre de lignes: 12907
Nombre de colonnes: 26
Doublons : 0



In [57]:
print("Les valeurs manquantes")
display(report["missing_values"])

Les valeurs manquantes


,missing_count,missing_percentage
date,0,0.000000
service,0,0.000000
gare_depart,0,0.000000
gare_arrivee,0,0.000000
duree_moyenne,0,0.000000
nb_train_prevu,0,0.000000
nb_annulation,0,0.000000
commentaire_annulation,12907,100.000000
nb_train_depart_retard,0,0.000000
retard_moyen_depart,0,0.000000


Nous observons que la plupart des colonnes sont renseignées. Nous avons aucune information sur "commentaire_annulation" et "commentaire_retards_depart" car 100% des données sont manquantes. Il faut encore étudier la colonne "commentaires_retard_arrivee" pour laquelle il manque 95% des valeurs. 

## 4.  Contrôle des identifiants

In [58]:
print("Vérification des identifiants")
display(report["identifiers"])

# format de date à respecter AAAA-MM
# doublon_metier : vérifie les doublons sur les colonnes date, service, gare_depart, gare_arrivee

Vérification des identifiants


,regle,nombre_anomalies
0,date_invalide,0
1,service_manquant,0
2,gare_depart_manquant,0
3,gare_arrivee_manquant,0
4,doublon_metier,0


## 5. Cohérence des circulations, des annulations et des retards

In [59]:
print("Vérification des comptages de trains")
display(report["train_counts"])

"""
    comptage_manquant_ou_invalide        valeurs manquantes ou invalides (NaN, None, pd.NA)
    comptage_negatif                     valeurs négatives
    comptage_non_entier                  valeurs non entières
    annulations_excessives               nombre d'annulations dépasse le nombre de trains prévus
    retards_depart_excessifs             nombre de retards au départ dépasse les trains ayant circulé
    retards_arrivee_excessifs            nombre de retards à l'arrivée dépasse les trains ayant circulé
    retards_15_excessifs                 nombre de retards supérieurs à 15 minutes dépasse les trains ayant circulé
    retards_30_excessifs                 nombre de retards supérieurs à 30 minutes dépasse les trains ayant circulé
    retards_60_excessifs                 nombre de retards supérieurs à 60 minutes dépasse les trains ayant circulé
    seuil_60_superieur_30                nombre de retards supérieurs à 60 minutes dépasse celui des retards supérieurs à 30 minutes
"""

Vérification des comptages de trains


,regle,nombre_anomalies
0,comptage_manquant_ou_invalide,0
1,comptage_negatif,43
2,comptage_non_entier,0
3,annulations_excessives,63
4,retards_depart_excessifs,64
5,retards_arrivee_excessifs,63
6,retards_15_excessifs,65
7,retards_30_excessifs,63
8,retards_60_excessifs,63
9,seuil_60_superieur_30,354


"\n    comptage_manquant_ou_invalide        valeurs manquantes ou invalides (NaN, None, pd.NA)\n    comptage_negatif                     valeurs négatives\n    comptage_non_entier                  valeurs non entières\n    annulations_excessives               nombre d'annulations dépasse le nombre de trains prévus\n    retards_depart_excessifs             nombre de retards au départ dépasse les trains ayant circulé\n    retards_arrivee_excessifs            nombre de retards à l'arrivée dépasse les trains ayant circulé\n    retards_15_excessifs                 nombre de retards supérieurs à 15 minutes dépasse les trains ayant circulé\n    retards_30_excessifs                 nombre de retards supérieurs à 30 minutes dépasse les trains ayant circulé\n    retards_60_excessifs                 nombre de retards supérieurs à 60 minutes dépasse les trains ayant circulé\n    seuil_60_superieur_30                nombre de retards supérieurs à 60 minutes dépasse celui des retards supérieurs à 30

Nous remarquons que le nombres d'anomalies est très similaire selon les différentes règles fixées, autour de 63.

## 6. Durées moyennes et pourcentages des causes de retard

In [60]:
print("Vérification des durées")
display(report["durations"])

"""
    duree_manquante_ou_non_numerique     valeurs manquantes ou non numériques (NaN, None, pd.NA, str, ...)
    duree_infinie                        durée infinie (np.inf, -np.inf)
    trajet_duree_invalide                durée négative ou nulle
    trajet_duree_non_entiere             durée non entière
    moyenne_trains_retardes_negative     Au moins une moyenne des trains retardés est négative à l'arrivée ou au départ
"""

Vérification des durées


,regle,nombre_anomalies
0,duree_manquante_ou_non_numerique,0
1,duree_infinie,0
2,trajet_duree_invalide,73
3,trajet_duree_non_entiere,0
4,moyenne_trains_retardes_negative,2


"\n    duree_manquante_ou_non_numerique     valeurs manquantes ou non numériques (NaN, None, pd.NA, str, ...)\n    duree_infinie                        durée infinie (np.inf, -np.inf)\n    trajet_duree_invalide                durée négative ou nulle\n    trajet_duree_non_entiere             durée non entière\n    moyenne_trains_retardes_negative     Au moins une moyenne des trains retardés est négative à l'arrivée ou au départ\n"

73 trajets possèdent des durées négatives ou nulle ce qui n'est pas cohérent.

In [61]:
print("Vérification des pourcentages")
display(report["percentages"])

Vérification des pourcentages


,regle,nombre_anomalies
0,pourcentage_manquant,0
1,pourcentage_non_numerique,0
2,pourcentage_negatif,0
3,pourcentage_superieur_100,0
4,somme_causes_differente_100,282


La somme de certains pourcentages est différente de 100. Il faudra vérifier si l'écart est import ou de quelques dixiemes.

In [62]:
print("Vérification des commentaires")
display(report["comments"])

Vérification des commentaires


,colonne,nb_manquants,nb_vides,nb_renseignes,pourcentage_renseignes
0,commentaire_annulation,12907,0,0,0.00
1,commentaire_retards_depart,12907,0,0,0.00
2,commentaires_retard_arrivee,12209,0,698,5.41


Comme préalablement remarqué très peu de commentaires sont renseignés.

## 7. Identification des observations atypiques ou incohérentes

### 7.1. Analyse des durées moyennes invalides

Nous cherchons à identifier les observations dont la durée moyenne du trajet est inférieure ou égale à zéro.

Une durée de trajet nulle ou négative est incohérente
pour un trajet en train. Nous vérifierons si ces valeurs
correspondent à des erreurs, à des valeurs particulières
ou à des observations sans circulation.

In [63]:

# Sélectionner les observations dont la durée moyenne
# du trajet est inférieure ou égale à zéro.
durations_invalides = data.loc[
    data["duree_moyenne"] <= 0,
    [
        "date",
        "service",
        "gare_depart",
        "gare_arrivee",
        "duree_moyenne",
        "nb_train_prevu",
        "nb_annulation",
    ],
]

# Compter les durées négatives et nulles séparément.
print(
    "Durées négatives :",
    (data["duree_moyenne"] < 0).sum(),
)

print(
    "Durées nulles :",
    (data["duree_moyenne"] == 0).sum(),
)

display(durations_invalides.head(50))


Durées négatives : 0
Durées nulles : 73


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation
3406,2020-04,National,PARIS MONTPARNASSE,ST MALO,0,0,8
3407,2020-04,National,PARIS MONTPARNASSE,TOURS,0,0,9
3409,2020-04,National,NANTES,STRASBOURG,0,0,2
3411,2020-04,National,PARIS NORD,DOUAI,0,0,12
3412,2020-04,National,DUNKERQUE,PARIS NORD,0,0,15
3414,2020-04,National,MARSEILLE ST CHARLES,TOURCOING,0,0,1
3415,2020-04,National,LYON PART DIEU,MARNE LA VALLEE,0,0,19
3416,2020-04,National,MARNE LA VALLEE,MARSEILLE ST CHARLES,0,0,14
3417,2020-04,National,MARSEILLE ST CHARLES,MARNE LA VALLEE,0,0,13
3418,2020-04,National,PARIS LYON,AIX EN PROVENCE TGV,0,0,17


Les durées invalides sont donc toutes nulles et non négatives. Les 50 premières lignes sont du même mois, avril 2020, cela peut potentiellement être en lien avec le confinement en France dû au Covid.

Vérifions maintenant la proportion des durées invalides par mois :

In [64]:

# Compter les durées nulles par mois.
repartition_mensuelle = (
    durations_invalides.groupby("date")
    .size()
    .reset_index(name="nombre_durees_nulles")
    .sort_values("date")
)

display(repartition_mensuelle)

# Vérifier les caractéristiques des observations concernées.
print(
    "Observations avec zéro train prévu :",
    (durations_invalides["nb_train_prevu"] == 0).sum(),
)

print(
    "Observations avec des annulations positives :",
    (durations_invalides["nb_annulation"] > 0).sum(),
)

print(
    "Observations avec zéro train prévu ET des annulations :",
    (
        (durations_invalides["nb_train_prevu"] == 0)
        & (durations_invalides["nb_annulation"] > 0)
    ).sum(),
)


,date,nombre_durees_nulles
0,2020-04,50
1,2020-05,9
2,2020-06,4
3,2020-08,2
4,2020-09,2
5,2020-10,2
6,2020-11,2
7,2020-12,2


Observations avec zéro train prévu : 73
Observations avec des annulations positives : 63
Observations avec zéro train prévu ET des annulations : 63



### Interprétation des durées moyennes nulles

L'analyse des durées moyennes révèle 73 observations présentant
une durée nulle, soit environ 0,57 % du jeu de données.
Aucune durée négative n'a été détectée.

#### Durées invalides par mois

Les 73 observations sont exclusivement concentrées en 2020,
entre avril et décembre, avec une forte concentration en
avril (50 observations, soit 68,5 %).

Cette période correspond très certainement aux perturbations exceptionnelles
du trafic ferroviaire pendant la pandémie du Covid.

#### Relation avec les circulations

Les 73 observations présentent toutes un nombre de circultaions prévues égal à zéro.

Parmi elles, 63 présentent néanmoins un nombre
d'annulations strictement positif.

Ces résultats suggèrent que les durées nulles sont liées
à des situations particulières d'exploitation, plutôt
qu'à des durées réelles de trajet.

Cependant, la documentation des variables ne permet
pas encore de confirmer précisément la convention
utilisée pour ces observations.


### 7.2. Analyse des incohérences liées aux annulations

Le contrôle des comptages a identifié 63 observations pour
lesquelles le nombre d'annulations dépasse le nombre de
circulations prévues.

Lors de l'analyse des durées moyennes, nous avons également
constaté que 63 des 73 observations présentant une durée
nulle possèdent un nombre d'annulations positif.

Nous cherchons donc à déterminer si ces deux anomalies
concernent exactement les mêmes observations.

L'objectif est de comprendre si les incohérences détectées
sont indépendantes ou si elles résultent d'une même
particularité des données SNCF.

Cette vérification est également importante pour la
construction de notre future variable cible en
Machine Learning.


In [65]:

# Sélectionner les observations pour lesquelles le nombre
# d'annulations dépasse le nombre de trains prévus.
annulations_excessives = data.loc[
    data["nb_annulation"] > data["nb_train_prevu"]
].copy()

# Calculer le nombre total d'observations concernées.
print(
    "Nombre d'observations :",
    len(annulations_excessives),
)

# Analyser la répartition mensuelle des anomalies.
repartition_annulations = (
    annulations_excessives.groupby("date")
    .size()
    .reset_index(name="nombre_anomalies")
    .sort_values("date")
)

display(repartition_annulations)

# Vérifier les liens avec les autres anomalies.
print(
    "Durée moyenne nulle :",
    (annulations_excessives["duree_moyenne"] == 0).sum(),
)

print(
    "Nombre de trains prévus nul :",
    (annulations_excessives["nb_train_prevu"] == 0).sum(),
)

print(
    "Retards à l'arrivée positifs :",
    (
        annulations_excessives["nb_train_retard_arrivee"] > 0
    ).sum(),
)

# Examiner les premières observations concernées.
display(
    annulations_excessives[
        [
            "date",
            "service",
            "gare_depart",
            "gare_arrivee",
            "nb_train_prevu",
            "nb_annulation",
            "duree_moyenne",
            "nb_train_retard_arrivee",
        ]
    ].head(15)
)


Nombre d'observations : 63


,date,nombre_anomalies
0,2020-04,50
1,2020-05,9
2,2020-06,4


Durée moyenne nulle : 63
Nombre de trains prévus nul : 63
Retards à l'arrivée positifs : 0


,date,service,gare_depart,gare_arrivee,nb_train_prevu,nb_annulation,duree_moyenne,nb_train_retard_arrivee
3406,2020-04,National,PARIS MONTPARNASSE,ST MALO,0,8,0,0
3407,2020-04,National,PARIS MONTPARNASSE,TOURS,0,9,0,0
3409,2020-04,National,NANTES,STRASBOURG,0,2,0,0
3411,2020-04,National,PARIS NORD,DOUAI,0,12,0,0
3412,2020-04,National,DUNKERQUE,PARIS NORD,0,15,0,0
3414,2020-04,National,MARSEILLE ST CHARLES,TOURCOING,0,1,0,0
3415,2020-04,National,LYON PART DIEU,MARNE LA VALLEE,0,19,0,0
3416,2020-04,National,MARNE LA VALLEE,MARSEILLE ST CHARLES,0,14,0,0
3417,2020-04,National,MARSEILLE ST CHARLES,MARNE LA VALLEE,0,13,0,0
3418,2020-04,National,PARIS LYON,AIX EN PROVENCE TGV,0,17,0,0



#### Interprétation des anomalies d'annulation

On observe **63 anomalies d'annulation (0,49 % des données)**, toutes situées entre avril et juin 2020, principalement en avril (50 observations). Ces lignes présentent toutes un nombre de trains prévus et une durée moyenne nuls, mais des annulations positives. Aucun retard à l'arrivée n'est enregistré.

Ces anomalies semblent liées aux perturbations du trafic pendant le Covid-19, sans qu'on puisse en confirmer précisément la cause. On choisit donc de **conserver ces observations dans les données brutes**, mais de les traiter à part lors des analyses et de ne pas calculer de taux de retard lorsque le dénominateur est nul ou négatif.



### 7.3. Analyse des incohérences entre les seuils de retard

On observe 354 lignes où le nombre de retards supérieurs à
60 minutes dépasse celui des retards supérieurs à 30 minutes.

Cette situation semble incohérente puisque tout retard
supérieur à 60 minutes est également supérieur à 30 minutes.

Nous allons regarder si ces anomalies sont concentrées sur
certaines années, certains services ou certaines liaisons,
puis mesurer l'importance des écarts.


In [66]:

# Sélectionner les observations où le nombre de retards
# de plus de 60 min dépasse celui des retards de plus de 30 min.
seuils_incoherents = data.loc[
    data["nb_train_retard_sup_60"]
    > data["nb_train_retard_sup_30"]
].copy()

# Calculer l'écart entre les deux indicateurs.
seuils_incoherents["ecart"] = (
    seuils_incoherents["nb_train_retard_sup_60"]
    - seuils_incoherents["nb_train_retard_sup_30"]
)

# Extraire l'année à partir de la date.
seuils_incoherents["annee"] = (
    seuils_incoherents["date"].str[:4]
)

print("Nombre d'observations :", len(seuils_incoherents))

# Répartition des anomalies par année.
print("\nRépartition par année :")
display(
    seuils_incoherents.groupby("annee")
    .size()
    .reset_index(name="nombre_anomalies")
)

# Répartition par service.
print("\nRépartition par service :")
display(
    seuils_incoherents.groupby("service")
    .size()
    .reset_index(name="nombre_anomalies")
)

# Statistiques descriptives sur les écarts.
print("\nStatistiques des écarts :")
display(seuils_incoherents["ecart"].describe())

# Identifier les liaisons les plus concernées.
print("\nLiaisons les plus concernées :")
display(
    seuils_incoherents.groupby(
        ["gare_depart", "gare_arrivee"]
    )
    .size()
    .reset_index(name="nombre_anomalies")
    .sort_values("nombre_anomalies", ascending=False)
    .head(10)
)

# Afficher les 15 plus grands écarts.
print("\nObservations avec les plus grands écarts :")
display(
    seuils_incoherents[
        [
            "date",
            "service",
            "gare_depart",
            "gare_arrivee",
            "nb_train_prevu",
            "nb_train_retard_sup_30",
            "nb_train_retard_sup_60",
            "ecart",
        ]
    ]
    .sort_values("ecart", ascending=False)
    .head(15)
)


Nombre d'observations : 354

Répartition par année :


,annee,nombre_anomalies
0,2025,354



Répartition par service :


,service,nombre_anomalies
0,International,40
1,National,314



Statistiques des écarts :


count    354.000000
mean      14.429379
std       10.562472
min        1.000000
25%        6.000000
50%       12.000000
75%       20.000000
max       69.000000
Name: ecart, dtype: float64


Liaisons les plus concernées :


,gare_depart,gare_arrivee,nombre_anomalies
0,AIX EN PROVENCE TGV,PARIS LYON,3
1,ANGERS SAINT LAUD,PARIS MONTPARNASSE,3
2,ANGOULEME,PARIS MONTPARNASSE,3
3,ANNECY,PARIS LYON,3
4,ARRAS,PARIS NORD,3
5,AVIGNON TGV,PARIS LYON,3
6,BARCELONA,PARIS LYON,3
7,BELLEGARDE (AIN),PARIS LYON,3
8,BESANCON FRANCHE COMTE TGV,PARIS LYON,3
9,BORDEAUX ST JEAN,PARIS MONTPARNASSE,3



Observations avec les plus grands écarts :


,date,service,gare_depart,gare_arrivee,nb_train_prevu,nb_train_retard_sup_30,nb_train_retard_sup_60,ecart
10355,2025-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,1029,0,69,69
10380,2025-01,National,LYON PART DIEU,LILLE,352,-44,21,65
10439,2025-01,National,PARIS MONTPARNASSE,BORDEAUX ST JEAN,1020,0,55,55
10574,2025-03,National,LYON PART DIEU,LILLE,343,-11,40,51
10339,2025-01,National,NANTES,PARIS MONTPARNASSE,632,1,50,49
10384,2025-01,National,MARNE LA VALLEE,LYON PART DIEU,421,0,45,45
10334,2025-01,National,LILLE,LYON PART DIEU,383,0,42,42
10424,2025-01,National,LYON PART DIEU,MARSEILLE ST CHARLES,502,-2,39,41
10419,2025-01,National,ANGERS SAINT LAUD,PARIS MONTPARNASSE,523,-12,28,40
10393,2025-01,National,RENNES,PARIS MONTPARNASSE,823,0,38,38



#### Interprétation des incohérences entre les seuils

On observe **354 incohérences (2,74 % des données)**, toutes situées en 2025, où le nombre de retards supérieurs à 60 minutes dépasse celui des retards supérieurs à 30 minutes.

Les écarts sont parfois importants (14,43 trains en moyenne, jusqu'à 69). Certaines valeurs du nombre de retards supérieurs à 30 minutes sont même négatives, ce qui est impossible pour un comptage de trains.

Ces anomalies semblent donc provenir d'un problème de données sur une période précise, plutôt que de certaines liaisons. **On conserve les observations brutes, mais on évitera d'utiliser les valeurs suspectes pour les analyses et le ML tant qu'elles n'auront pas été vérifiées.**



### 7.4. Analyse des pourcentages des causes de retard

On observe 282 lignes pour lesquelles la somme des six causes
de retard s'écarte de 100 %.

Nous allons vérifier si ces écarts proviennent simplement
des arrondis ou s'ils révèlent des problèmes de données.

On regardera notamment leur répartition dans le temps
et l'importance des écarts.


In [67]:

# Sélectionner les six colonnes des causes de retard.
colonnes_causes = [
    colonne
    for colonne in data.columns
    if colonne.startswith("prct_cause_")
]

# Convertir les pourcentages en valeurs numériques.
causes = data[colonnes_causes].apply(
    pd.to_numeric,
    errors="coerce",
)

# Calculer la somme uniquement si les six valeurs existent.
somme_causes = causes.sum(
    axis=1,
    min_count=len(colonnes_causes),
)

# Calculer l'écart absolu par rapport à 100 %.
ecart_causes = (somme_causes - 100).abs()

# Vérifier que les valeurs sont comprises entre 0 et 100.
bornes_valides = causes.ge(0).all(axis=1) & causes.le(100).all(axis=1)

# Sélectionner les observations avec un écart supérieur
# à 0,1 point, en conservant les lignes complètes et valides.
masque_anomalies = (
    causes.notna().all(axis=1)
    & bornes_valides
    & ecart_causes.gt(0.1)
)

causes_incoherentes = data.loc[masque_anomalies].copy()

causes_incoherentes["somme_causes"] = somme_causes.loc[
    masque_anomalies
]
causes_incoherentes["ecart"] = ecart_causes.loc[
    masque_anomalies
]

print("Nombre d'observations :", len(causes_incoherentes))

# Répartition annuelle.
print("\nRépartition par année :")
display(
    causes_incoherentes.assign(
        annee=causes_incoherentes["date"].str[:4]
    )
    .groupby("annee")
    .size()
    .reset_index(name="nombre_anomalies")
)

# Statistiques sur l'amplitude des écarts.
print("\nStatistiques des écarts :")
display(causes_incoherentes["ecart"].describe())

# Afficher les observations avec les plus grands écarts.
print("\nObservations les plus incohérentes :")
display(
    causes_incoherentes[
        ["date", "service", "gare_depart", "gare_arrivee",
         "somme_causes", "ecart"]
    ]
    .sort_values("ecart", ascending=False)
    .head(15)
)


Nombre d'observations : 282

Répartition par année :


,annee,nombre_anomalies
0,2018,74
1,2019,9
2,2020,110
3,2021,27
4,2022,19
5,2023,5
6,2024,18
7,2025,6
8,2026,14



Statistiques des écarts :


count    282.000000
mean      96.639122
std       17.590744
min        0.877193
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: ecart, dtype: float64


Observations les plus incohérentes :


,date,service,gare_depart,gare_arrivee,somme_causes,ecart
12807,2026-09,National,TOURCOING,BORDEAUX ST JEAN,0.0,100.0
318,2018-03,National,MARSEILLE ST CHARLES,TOURCOING,0.0,100.0
389,2018-03,International,MARSEILLE ST CHARLES,MADRID,0.0,100.0
421,2018-04,National,MARSEILLE ST CHARLES,TOURCOING,0.0,100.0
495,2018-04,International,MARSEILLE ST CHARLES,MADRID,0.0,100.0
508,2018-04,National,RENNES,PARIS VAUGIRARD,0.0,100.0
536,2018-05,National,BELLEGARDE (AIN),PARIS LYON,0.0,100.0
546,2018-05,International,PARIS LYON,ITALIE,0.0,100.0
10446,2025-02,International,PARIS LYON,ITALIE,0.0,100.0
10370,2025-01,International,PARIS LYON,ITALIE,0.0,100.0


hypothèse: lorsque les 6 causes valent 0, cela pourrait indiquer qu'acuun retard pertinent pour cette répartition n'a été enregistré, ou que les causes n'ont pas été renseignées. Il faut vérifier cette hypothèse avant de considérer ces lignes comme incorrectes.

In [68]:

# Séparer les observations selon leur somme des causes.
causes_somme_nulle = causes_incoherentes.loc[
    causes_incoherentes["somme_causes"] == 0
].copy()

causes_somme_non_nulle = causes_incoherentes.loc[
    causes_incoherentes["somme_causes"] != 0
].copy()

print("Sommes égales à 0 % :", len(causes_somme_nulle))
print(
    "Autres sommes incohérentes :",
    len(causes_somme_non_nulle),
)

# Vérifier si des retards sont enregistrés
# lorsque toutes les causes valent zéro.
print(
    "Sommes nulles avec retards à l'arrivée :",
    (causes_somme_nulle["nb_train_retard_arrivee"] > 0).sum(),
)

print(
    "Sommes nulles avec retards > 15 min :",
    (causes_somme_nulle["nb_train_retard_sup_15"] > 0).sum(),
)

# Examiner les sommes non nulles qui ne valent pas 100 %.
display(
    causes_somme_non_nulle[
        [
            "date",
            "gare_depart",
            "gare_arrivee",
            "nb_train_retard_arrivee",
            "somme_causes",
            "ecart",
        ]
    ].sort_values("ecart", ascending=False).head(15)
)


Sommes égales à 0 % : 272
Autres sommes incohérentes : 10
Sommes nulles avec retards à l'arrivée : 80
Sommes nulles avec retards > 15 min : 80


,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,somme_causes,ecart
12512,2026-06,PARIS MONTPARNASSE,LAVAL,65,84.615385,15.384615
12514,2026-06,PARIS MONTPARNASSE,QUIMPER,46,86.956522,13.043478
12487,2026-06,PARIS MONTPARNASSE,VANNES,56,87.500000,12.500000
12672,2026-08,LILLE,LYON PART DIEU,41,97.560976,2.439024
12886,2026-09,VALENCE ALIXAN TGV,PARIS LYON,49,97.959184,2.040816
12480,2026-06,MARSEILLE ST CHARLES,LILLE,62,98.387097,1.612903
12814,2026-09,LILLE,LYON PART DIEU,66,98.484848,1.515152
12692,2026-08,MARSEILLE ST CHARLES,LILLE,69,98.550725,1.449275
12758,2026-08,PARIS MONTPARNASSE,ST PIERRE DES CORPS,73,98.630137,1.369863
12430,2026-06,LYON PART DIEU,LILLE,114,99.122807,0.877193



#### Interprétation des pourcentages des causes de retard

On observe **282 anomalies (2,18 % des données)**, dont 272 correspondent à une somme des causes égale à 0 %. Parmi ces dernières, 80 observations présentent pourtant des retards, ce qui suggère que les causes ne sont pas toujours renseignées.

Les 10 autres anomalies, toutes situées en 2026, présentent des sommes comprises entre 84.6% et 99.1%. Certains écarts sont donc trop importants pour être expliqués uniquement par des arrondis.

Comme les observations précédentes on choisit de **conserver les observations brutes**, mais de considérer les répartitions des causes comme non exploitables lorsque leur somme ne vaut pas approximativement 100 %. Les valeurs nulles ne doivent pas être automatiquement interprétées comme une absence de retard.



## 8. Conclusion

Cette première analyse nous a permis d'évaluer la qualité des données SNCF et d'identifier les principales limites à prendre en compte avant les analyses statistiques et la modélisation.

Le jeu de données contient des observations particulières, notamment en 2020 et début 2025, ainsi que des informations incomplètes sur les causes des retards. Ces problèmes ne remettent pas nécessairement en cause l'ensemble du jeu de données, mais nécessitent des traitements adaptés.

La prochaine étape consistera à préparer une version exploitable des données, en conservant une trace des modifications effectuées. Nous pourrons ensuite étudier les différences de régularité entre les liaisons et construire nos premiers modèles prédictifs, avant d'évaluer l'apport des caractéristiques du réseau TGV.



# III. Transmission des résultats à l'ETL

| Anomalie | Observations | Recommandation ETL |
|---|---:|---|
| Durées nulles | 73 | Ne pas utiliser ces valeurs pour analyser la durée des trajets. |
| Annulations supérieures aux trains prévus | 63 | Vérifier les conventions de comptage et les dénominateurs. |
| Incohérences entre retards > 30 et > 60 min | 354 | Vérifier les comptages du premier trimestre 2025. |
| Comptages négatifs | 43 | Signaler les valeurs comme invalides avant utilisation. |
| Sommes des causes nulles | 272 | Distinguer absence de causes renseignées et absence de retard. |
| Autres sommes des causes incorrectes | 10 | Vérifier les pourcentages avant leur exploitation. |

Les anomalies peuvent concerner plusieurs contrôles
simultanément. Le nombre total d'observations signalées
ne correspond donc pas forcément à la somme de ces lignes.

On choisit de conserver les données brutes et de produire
un fichier séparé contenant les observations à examiner.


In [69]:

# Créer un tableau de suivi sans modifier les données brutes.
suivi_qualite = data[
    [
        "date",
        "service",
        "gare_depart",
        "gare_arrivee",
        "nb_train_prevu",
        "nb_annulation",
        "duree_moyenne",
        "nb_train_retard_arrivee",
        "nb_train_retard_sup_30",
        "nb_train_retard_sup_60",
    ]
].copy()

# Garder une référence à la ligne d'origine pour
# retrouver facilement l'observation dans le dataset.
suivi_qualite.insert(0, "index_source", data.index)

# Créer un indicateur pour chaque type d'anomalie.
# True signifie que l'observation doit être examinée.
suivi_qualite["duree_nulle"] = (
    data["duree_moyenne"] == 0
)

suivi_qualite["annulations_excessives"] = (
    data["nb_annulation"] > data["nb_train_prevu"]
)

suivi_qualite["seuils_incoherents"] = (
    data["nb_train_retard_sup_60"]
    > data["nb_train_retard_sup_30"]
)

suivi_qualite["comptage_30_negatif"] = (
    data["nb_train_retard_sup_30"] < 0
)

# Réutiliser les sommes des causes calculées en section 7.4.
suivi_qualite["somme_causes"] = somme_causes

suivi_qualite["causes_somme_nulle"] = (
    masque_anomalies & somme_causes.eq(0)
)

suivi_qualite["causes_autre_ecart"] = (
    masque_anomalies & somme_causes.ne(0)
)

# Lister les indicateurs utilisés dans le rapport.
colonnes_anomalies = [
    "duree_nulle",
    "annulations_excessives",
    "seuils_incoherents",
    "comptage_30_negatif",
    "causes_somme_nulle",
    "causes_autre_ecart",
]

# Une observation est signalée si au moins une
# des conditions précédentes est vérifiée.
suivi_qualite["anomalie_detectee"] = (
    suivi_qualite[colonnes_anomalies].any(axis=1)
)

# Conserver uniquement les observations signalées.
anomalies_etl = suivi_qualite.loc[
    suivi_qualite["anomalie_detectee"]
].copy()

# Créer le dossier de sortie.
dossier_rapport = project_root / "data" / "rapports"
dossier_rapport.mkdir(parents=True, exist_ok=True)

# Exporter le rapport avec un séparateur compatible CSV.
chemin_rapport = dossier_rapport / "anomalies_qualite.csv"

anomalies_etl.to_csv(
    chemin_rapport,
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

print("Nb anomalies:", len(anomalies_etl))


Nb anomalies: 636
